<a href="https://colab.research.google.com/github/hwangho-kim/Transformer_Fewshot_PdM/blob/main/RUL_%EC%98%88%EC%B8%A1_%EB%B0%8F_%EC%8B%9C%EA%B0%81%ED%99%94_%EC%BD%94%EB%93%9C_VTR_R06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.linear_model import LinearRegression
import warnings

warnings.filterwarnings('ignore')

# 한글 폰트 설정 (터미널 출력용)
import platform
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin': # Mac
    plt.rc('font', family='AppleGothic')
else:
    plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False


def generate_dummy_data(file_path):
    """테스트를 위한 가상 시계열 센서 데이터 생성 (하나의 사이클만 생성)"""
    print("Generating virtual data...")
    np.random.seed(42)

    # 최근 30일 간의 데이터라고 가정
    dates = pd.date_range(end=pd.Timestamp.now(), periods=1000, freq='1h')
    mean_values = []

    current_val = 200
    for i in range(1000):
        # 지속적인 상승 추세 (열화 현상)
        current_val += np.random.normal(0.8, 2.0)
        mean_values.append(current_val)

    df = pd.DataFrame({'start_time': dates, 'mean': mean_values})
    df.to_parquet(file_path)
    print(f"Virtual data saved: {file_path}")


def predict_failure_time(parquet_path, threshold=1100):
    """현재 추세를 기반으로 임계치에 도달하는 예상 시점(년/월/일) 예측"""
    print("\n1. Loading Parquet data...")
    df = pd.read_parquet(parquet_path)

    if 'start_time' not in df.columns or 'mean' not in df.columns:
        raise ValueError("The data must contain 'start_time' and 'mean' columns.")

    df['start_time'] = pd.to_datetime(df['start_time'])
    df = df.sort_values('start_time').reset_index(drop=True)

    print("2. Analyzing degradation trend (Linear Regression)...")
    # 교체 이력이 없는 단일 추세 데이터이므로, 전체 데이터를 바탕으로 추세를 분석합니다.

    # start_time을 초 단위의 숫자형(Timestamp)으로 변환
    X = df['start_time'].astype('int64') // 10**9
    X_reshaped = X.values.reshape(-1, 1)
    y = df['mean'].values

    # 선형 회귀 모델 학습 (시간에 따른 mean 값의 상승 기울기 계산)
    model = LinearRegression()
    model.fit(X_reshaped, y)

    coef = model.coef_[0]
    intercept = model.intercept_

    if coef <= 0:
        print("Warning: The 'mean' value trend is decreasing or flat. Cannot predict threshold reach time.")
        return df, None, model

    # [예상 시점 역산] y = coef * x + intercept  =>  x = (y - intercept) / coef
    predicted_timestamp = (threshold - intercept) / coef
    predicted_time = pd.to_datetime(predicted_timestamp, unit='s')

    # 결과 출력 (년 월 일 시 분 초)
    print("\n==================================================")
    print(f" 🎯 1100 도달 예상 시점 : {predicted_time.strftime('%Y년 %m월 %d일 %H시 %M분 %S초')}")
    print("==================================================\n")

    return df, predicted_time, model


def visualize_prediction(df, predicted_time, model, threshold=1100):
    """데이터 및 도달 예상 시점 시각화"""
    if predicted_time is None:
        return

    print("3. Generating Visualization Chart...")
    plt.figure(figsize=(12, 7))

    # 실제 데이터 플롯
    plt.plot(df['start_time'], df['mean'], color='#1f77b4', alpha=0.7, label='Actual Sensor Data (mean)')

    # 선형 회귀 추세선 (시작점부터 예상 시점까지)
    X_start = df['start_time'].astype('int64').min() // 10**9
    X_end = int(predicted_time.timestamp())

    trend_x = pd.to_datetime([X_start, X_end], unit='s')
    trend_y = model.predict(np.array([X_start, X_end]).reshape(-1, 1))

    plt.plot(trend_x, trend_y, color='#ff7f0e', linestyle='--', linewidth=2.5, label='Degradation Trend Line')

    # 임계치 및 예상 시점 마킹
    plt.axhline(y=threshold, color='red', linestyle='-', linewidth=1.5, alpha=0.8, label=f'Threshold ({threshold})')
    plt.axvline(x=predicted_time, color='gray', linestyle=':', linewidth=1.5)

    # 도달 시점 점 찍기
    plt.scatter([predicted_time], [threshold], color='red', s=100, zorder=5)

    # 차트에 예상 시점 텍스트 표시
    date_str = predicted_time.strftime('%Y-%m-%d %H:%M:%S')
    plt.annotate(f'Predicted:\n{date_str}',
                 xy=(predicted_time, threshold),
                 xytext=(-100, 20), textcoords='offset points',
                 arrowprops=dict(arrowstyle="->", connectionstyle="arc3,rad=.2", color='black'),
                 fontsize=12, fontweight='bold', color='red')

    # 차트 꾸미기 (모두 영어 처리)
    plt.title('Prediction of Time to Reach Sensor Threshold', fontsize=16, fontweight='bold')
    plt.xlabel('Date / Time', fontsize=12)
    plt.ylabel('Sensor Value (mean)', fontsize=12)

    # X축 시간 포맷 설정
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
    plt.xticks(rotation=45)

    plt.grid(True, linestyle=':', alpha=0.7)
    plt.legend(fontsize=12, loc='upper left')
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    parquet_file = 'sample_sensor_data_linear.parquet'

    # 1. 테스트용 파일이 없으면 생성
    if not os.path.exists(parquet_file):
        generate_dummy_data(parquet_file)

    try:
        # 2. 데이터 분석 및 예상 시점 계산
        current_cycle_data, pred_time, lr_model = predict_failure_time(parquet_file, threshold=1100)

        # 3. 차트 시각화
        visualize_prediction(current_cycle_data, pred_time, lr_model, threshold=1100)

    except Exception as e:
        print(f"Error occurred: {e}")